# Sales Assistant Chatbot - Zero-shot vs Few-shot Prompting (Gradio)

### BUSINESS CHALLENGE:

**CloudNova IT Solutions** is a fictional company that sells:
- **Cloud services on Azure only** (no AWS, no on-premises)
- **Software services only** (no hardware)
- **20% discount on all services**, and a **40% discount specifically on Azure**

The goal is to build a sales chatbot that answers customer questions and *proactively 
encourages the Azure discount* when relevant without inventing services we don't offer 
(like AWS or on-prem).

This notebook compares two prompting strategies for the same task:
1. **Zero-shot** - just an instruction, no examples
2. **Few-shot** - the instruction plus a handful of example conversations showing the 
   exact tone/behavior we want

Both run through **Gradio's `ChatInterface`**, which gives us chat history "for free" - 
every time the customer sends a message, Gradio hands our function the full conversation 
so far, so the bot stays consistent across turns.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

c:\Users\Mustafa Ansari\Downloads\llm engineering\llm-engineering\.venv\Lib\site-packages\huggingface_hub\constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


#### Step 1: Load the Groq API key

In [3]:
load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set - check your .env file")

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
MODEL = "llama-3.3-70b-versatile"

Groq API Key exists and begins gsk_ePDh


In [4]:
company_info = """
Company: CloudNova IT Solutions

What we offer:
- Cloud services on Microsoft Azure ONLY (we do NOT offer AWS, Google Cloud, or on-premises hosting)
- Software services ONLY (we do NOT sell or install hardware)

Discounts:
- 20% off on all our services
- 40% off specifically on Azure cloud services (our best deal - always worth mentioning)
"""

#### Step 2: Zero-shot system prompt

Just the instruction and the company info. No examples of *how* to respond - 
here I am trusting the model to figure out the right tone and sales behavior on its own.

In [23]:
zero_shot_system_prompt = f"""
You are a friendly sales assistant for CloudNova IT Solutions.
Answer customer questions and try to encourage them to buy, especially highlighting
the Azure discount when relevant.

{company_info}
"""

#### Step 3: Few-shot system prompt

Same instruction and company info, but now I add a few example exchanges showing 
exactly how we want the bot to behave - e.g. steering away from AWS/hardware questions, 
and always mentioning the 40% Azure discount when cloud comes up.

In [29]:
few_shot_examples = """
Example 1:
Customer: Do you offer AWS hosting?
Assistant: We don't offer AWS, but we're Azure specialists - and right now Azure services
are 40% off, which is our best deal. Want me to walk you through what's included? Also we don't provide SQL server.

Example 2:
Customer: Can you also supply the servers/hardware for our office?
Assistant: We're a software-and-cloud-only company, so we don't sell hardware. What we can
do is move that workload to Azure instead, which also qualifies for our 40% Azure discount.

Example 3:
Customer: What discounts do you have?
Assistant: All our services come with a 20% discount, and if you go with Azure cloud
services specifically, that jumps to 40% off - it's a great time to make the switch.
"""

few_shot_system_prompt = f"""
You are a friendly sales assistant for CloudNova IT Solutions.
Answer customer questions and try to encourage them to buy, especially highlighting
the Azure discount when relevant. Follow the style of the examples below closely.

{company_info}

Here are some example conversations showing the tone and behavior we want:
{few_shot_examples}
"""

#### Step 4: The chat function

This is the function Gradio calls every time the customer sends a message.

`gr.ChatInterface` automatically gives us two things:
- `message` - what the customer just typed
- `history` - the conversation so far, as a list of `[user_msg, bot_msg]` pairs - this 
  is our "callback": it lets the bot remember earlier turns without us writing any extra 
  memory code ourselves.

here it loops through the history, rebuild it into the role/content format the Groq API 
expects, prepend the system prompt, add the new message, and send it all off.


In [25]:
def chat(message, history, system_prompt):
    cleaned_history = [
        {"role": entry["role"], "content": entry["content"]}
        for entry in history
    ]
    messages = [{"role": "system", "content": system_prompt}] + cleaned_history + [{"role": "user", "content": message}]

    response = groq.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.3,
    )
    return response.choices[0].message.content

#### Step 5: Zero-shot chatbot

Launching the zero-shot version first. Try asking it:
- "Do you offer AWS?"
- "Can you install servers for us?"
- "What discounts do you have?"

In [26]:
def chat_zero_shot(message, history):
    return chat(message, history, zero_shot_system_prompt)

gr.ChatInterface(
    fn=chat_zero_shot,
    title="CloudNova Sales Assistant (Zero-shot)",
).launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


#### Step 6: Few-shot chatbot

Same questions, same model, same company info - the only difference is the few-shot 
examples in the system prompt. Compare the answers side by side with the zero-shot version.

In [28]:
def chat_few_shot(message, history):
    return chat(message, history, few_shot_system_prompt)

gr.ChatInterface(
    fn=chat_few_shot,
    title="CloudNova Sales Assistant (Few-shot)",
).launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
